# Lesson 3 Lab: Teaching Computers What Words Mean
### Word Embeddings, Similarity, Word Math, and Smart Search

In Lesson 2, we counted words. But counting cannot tell that "happy" and "glad" mean almost the same thing. Today, we give every word a **list of numbers that captures its meaning**. These lists are called **word embeddings**.

**What you will do:**
1. See why counting words fails at search
2. Explore real word embeddings
3. Draw a map of word meanings
4. Measure how similar words are
5. Solve puzzles with "word math"
6. Build a **smart search tool**
7. Test embeddings on the spam detector from Lesson 2
8. Discover **bias** hidden in embeddings
9. Practice exercises

**How to use this notebook:** Run each cell from top to bottom with **Shift + Enter**. Read the comments (lines starting with `#`). They explain what each line does.

---
## Step 0: Setup

We need **gensim**, a Python library for word embeddings. Colab already has the other libraries.

> **Note:** If you see an error after installing gensim in Colab, click **Runtime > Restart session**, then run the cells again from the top.

In [ ]:
# Install gensim (and the other libraries if you are on your own computer).
!pip install -q gensim scikit-learn pandas matplotlib

In [ ]:
import re                                   # for splitting text into words
import numpy as np                          # math with lists of numbers
import pandas as pd                         # tables of data
import matplotlib.pyplot as plt             # charts
import gensim.downloader as api             # downloads pretrained embeddings

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

print("Libraries are ready!")

### Download the GloVe embeddings
**GloVe** learned from about 6 billion words of Wikipedia and news text. It knows **400,000 words**, and each word has **50 numbers**. The download is about 66 MB, so it may take a minute.

In [ ]:
model = api.load("glove-wiki-gigaword-50")

print("Number of words the model knows:", len(model.key_to_index))
print("Numbers per word:", model.vector_size)

---
## Part 1: The Problem with Counting Words

Imagine an online store with these product reviews. A shopper searches for **"cheap"**.

In [ ]:
reviews = [
    "The shoes were affordable and comfortable.",
    "This phone is too expensive for students.",
    "Great price, I saved a lot of money.",
    "The delivery was very slow.",
    "The package arrived late and damaged.",
    "The teacher explained the lesson clearly.",
    "The food tasted delicious and fresh.",
    "The burger was tasty but cold.",
    "Battery life is amazing on this laptop.",
    "The screen cracked after one week.",
]

# Use TF-IDF from Lesson 2 to search the reviews.
tfidf = TfidfVectorizer()
review_numbers = tfidf.fit_transform(reviews)

query_numbers = tfidf.transform(["cheap"])
scores = cosine_similarity(query_numbers, review_numbers)[0]   # one score per review

for score, review in zip(scores, reviews):
    print(f"{score:.2f}  {review}")

# Every score is 0.00! No review contains the exact word "cheap",
# so TF-IDF finds nothing, even though some reviews are clearly about low prices.

---
## Part 2: Meet Word Embeddings

A **word embedding** is a list of numbers that represents a word's meaning.

In [ ]:
# Look up the embedding for "king".
king = model["king"]

print("Length of the list:", len(king))
print("First 10 numbers:", king[:10].round(2))

In [ ]:
# Not every word is in the model. Always check first with "in".
for word in ["pizza", "basketball", "fr33", "jeepney"]:
    print(f"{word:<12} in the model? {word in model}")

### 2.1 A made-up example to build intuition
Before using real embeddings, let's make our own tiny ones with 3 traits. These numbers are **made up**.

In [ ]:
# Traits: [is it alive?, can you eat it?, is it big?]
toy = {
    "cat":   np.array([0.9, 0.1, 0.2]),
    "dog":   np.array([0.9, 0.1, 0.3]),
    "pizza": np.array([0.0, 0.9, 0.2]),
    "bus":   np.array([0.0, 0.0, 0.9]),
}

# Cosine similarity: how much two arrows point the same way (up to 1).
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("cat vs dog:  ", round(cosine(toy["cat"], toy["dog"]), 2))
print("cat vs pizza:", round(cosine(toy["cat"], toy["pizza"]), 2))
print("cat vs bus:  ", round(cosine(toy["cat"], toy["bus"]), 2))

### 2.2 Cosine similarity with real embeddings
Our `cosine()` function gives the same answer as gensim's built-in `similarity()`.

In [ ]:
print("Our function:", round(cosine(model["cat"], model["dog"]), 4))
print("gensim:      ", round(model.similarity("cat", "dog"), 4))

---
## Part 3: The Word Map

### 3.1 Finding a word's neighbors

In [ ]:
# most_similar() finds the closest words on the "word map".
for word in ["pizza", "basketball", "school", "smart"]:
    neighbors = [w for w, score in model.most_similar(word, topn=5)]
    print(f"{word:<12} -> {neighbors}")

In [ ]:
# YOUR TURN: try any word you like!
word = "music"
for neighbor, score in model.most_similar(word, topn=8):
    print(f"{neighbor:<15} {score:.2f}")

### 3.2 Draw the map
Each word has 50 numbers, but a screen is flat. **PCA** is a tool that squashes the 50 numbers down to 2, keeping as much of the pattern as possible.

In [ ]:
groups = {
    "animals":  ["cat", "dog", "horse", "cow", "puppy"],
    "food":     ["pizza", "burger", "rice", "bread", "noodles"],
    "vehicles": ["car", "bus", "train", "truck", "bicycle"],
    "school":   ["teacher", "student", "classroom", "homework", "exam"],
}
colors = {"animals": "#EF5B5B", "food": "#E9A82B", "vehicles": "#3FA7D6", "school": "#6C63FF"}

# Collect every word and its 50 numbers.
words = [w for group in groups.values() for w in group]
vectors = np.array([model[w] for w in words])

# Squash 50 numbers down to 2 numbers (x and y).
points = PCA(n_components=2).fit_transform(vectors)

plt.figure(figsize=(10, 6))
for group, group_words in groups.items():
    for w in group_words:
        x, y = points[words.index(w)]
        plt.scatter(x, y, color=colors[group], s=60)
        plt.annotate(w, (x + 0.05, y + 0.05), fontsize=11)
plt.title("A map of word meanings (GloVe embeddings squashed to 2D)")
plt.show()

# The computer was never told the groups. It found them by itself!

---
## Part 4: Measuring Similarity

In [ ]:
pairs = [("cat", "dog"), ("happy", "glad"), ("king", "queen"),
         ("cheap", "affordable"), ("happy", "table"), ("cat", "car")]

for a, b in pairs:
    print(f"{a + ' & ' + b:<22} {model.similarity(a, b):.2f}")

### 4.1 Solve the warm-up puzzle
Which word does not belong: breakfast, lunch, dinner, school?

In [ ]:
print(model.doesnt_match(["breakfast", "lunch", "dinner", "school"]))
print(model.doesnt_match(["apple", "banana", "mango", "car"]))

### 4.2 Surprise! Opposites can look similar

In [ ]:
print("happy & sad:", round(model.similarity("happy", "sad"), 2))

print("\nClosest words to 'cheap':")
for word, score in model.most_similar("cheap", topn=4):
    print(f"  {word:<12} {score:.2f}")

# "expensive" is the closest word to "cheap"!
# Opposites appear in the same kinds of sentences ("the price is cheap / expensive"),
# so embeddings think they are related.

---
## Part 5: Word Math

Because words are lists of numbers, we can add and subtract them.

In [ ]:
# king - man + woman = ?
# positive = words to ADD, negative = words to SUBTRACT
answers = model.most_similar(positive=["king", "woman"], negative=["man"], topn=3)

for word, score in answers:
    print(word, round(score, 2))

In [ ]:
# A helper function to make word math easier to read.
# It answers: "a is to b as c is to ???"
def analogy(a, b, c):
    result = model.most_similar(positive=[b, c], negative=[a], topn=1)
    return result[0][0]     # the first answer's word

print("man : king   = woman : ", analogy("man", "king", "woman"))
print("france : paris = japan : ", analogy("france", "paris", "japan"))
print("philippines : manila = japan : ", analogy("philippines", "manila", "japan"))

In [ ]:
# YOUR TURN: invent your own word math. Predict the answer first!
# dog is to puppy as cat is to ???  (You might expect "kitten.")
print(analogy("dog", "puppy", "cat"))

# Was the answer what you expected? Word math works often, but not always.

---
## Part 6: Build a Smart Search Tool

To compare whole sentences, we turn each sentence into **one** list of numbers by **averaging** its word embeddings.

In [ ]:
def sentence_vector(text):
    # 1. Lowercase the text and split it into words.
    words = re.findall(r"[a-z']+", text.lower())
    # 2. Keep words that are NOT stop words and ARE in the model.
    words = [w for w in words if w not in ENGLISH_STOP_WORDS and w in model]
    # 3. If no words are left, return a list of zeros.
    if not words:
        return np.zeros(model.vector_size)
    # 4. Average the embeddings of the remaining words.
    return np.mean([model[w] for w in words], axis=0)

# Turn every review into one vector (one row per review).
review_vectors = np.array([sentence_vector(r) for r in reviews])
print("Review table size:", review_vectors.shape)   # 10 reviews x 50 numbers

In [ ]:
def smart_search(query, top=3):
    query_vector = sentence_vector(query)
    scores = cosine_similarity([query_vector], review_vectors)[0]
    best = np.argsort(scores)[::-1][:top]       # positions of the highest scores
    print(f"Search: '{query}'")
    for i in best:
        print(f"  {scores[i]:.2f}  {reviews[i]}")
    print()

smart_search("cheap")
smart_search("yummy meal")
smart_search("broken")
smart_search("late shipping")

> Look at the third result for **"cheap"**. "This phone is too expensive" shows up because opposites look similar. Smart search is helpful, but not perfect.

In [ ]:
# Compare TF-IDF and embeddings side by side for the same searches.
rows = []
for query in ["cheap", "yummy meal", "broken", "late shipping"]:
    tfidf_best = cosine_similarity(tfidf.transform([query]), review_numbers)[0].max()
    emb_scores = cosine_similarity([sentence_vector(query)], review_vectors)[0]
    rows.append({"Search": query,
                 "Top review (embeddings)": reviews[emb_scores.argmax()],
                 "Embedding score": round(emb_scores.max(), 2),
                 "Best TF-IDF score": round(tfidf_best, 2)})
pd.DataFrame(rows)

In [ ]:
# YOUR TURN: search the reviews with your own words.
smart_search("great food")

---
## Part 7: Experiment: Embeddings vs. Bag of Words for Spam

Do embeddings make a better spam detector than Lesson 2's Bag of Words? Let's test both on the same data.

In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
sms = pd.read_csv(url, sep="\t", header=None, names=["label", "message"])
print("Messages:", len(sms))

In [ ]:
# Same split as Lesson 2, so the comparison is fair.
X_train, X_test, y_train, y_test = train_test_split(
    sms["message"], sms["label"], test_size=0.25, random_state=42, stratify=sms["label"])

# Model A: Bag of Words + Naive Bayes (from Lesson 2)
bow = CountVectorizer()
nb_model = MultinomialNB().fit(bow.fit_transform(X_train), y_train)
pred_a = nb_model.predict(bow.transform(X_test))

# Model B: averaged embeddings + Logistic Regression
def message_vector(text):
    # For spam, we keep stop words, because short messages need every word.
    words = [w for w in re.findall(r"[a-z']+", text.lower()) if w in model]
    return np.mean([model[w] for w in words], axis=0) if words else np.zeros(50)

train_vecs = np.array([message_vector(t) for t in X_train])
test_vecs = np.array([message_vector(t) for t in X_test])
lr_model = LogisticRegression(max_iter=2000).fit(train_vecs, y_train)
pred_b = lr_model.predict(test_vecs)

results = []
for name, preds in [("Bag of Words + Naive Bayes", pred_a), ("Embeddings + Logistic Regression", pred_b)]:
    cm = confusion_matrix(y_test, preds, labels=["ham", "spam"])
    results.append({"Method": name,
                    "Accuracy": f"{accuracy_score(y_test, preds):.1%}",
                    "False alarms": cm[0, 1],
                    "Spam missed": cm[1, 0]})
pd.DataFrame(results)

> **What happened?** Averaging blurs strong spam clues like "claim" and "150p" into one fuzzy meaning. Embeddings are great for search, but simple counting works better for spam. **Lesson:** pick the tool that fits the task.

---
## Part 8: Bias and Limits

### 8.1 Embeddings can learn stereotypes

In [ ]:
print("doctor - man + woman =", analogy("man", "doctor", "woman"))
print("nurse - woman + man  =", analogy("woman", "nurse", "man"))

# Women can be doctors and men can be nurses.
# The model copied an unfair pattern from the human texts it read.

### 8.2 Whose language does the model know?

In [ ]:
for word in ["adobo", "jeepney"]:
    neighbors = [w for w, s in model.most_similar(word, topn=5)]
    print(f"{word:<10} -> {neighbors}")

# To this model, "adobo" is a Mexican chili sauce, not the Filipino dish!
# It learned mostly from English Wikipedia and news, not Filipino text.

### 8.3 One word, one meaning
The word "bank" gets the **same** numbers whether it means a river bank or a money bank. Its neighbors are mostly about money.

In [ ]:
print([w for w, s in model.most_similar("bank", topn=8)])

---
## Part 9: Practice Exercises

Write your code where you see `# YOUR CODE HERE`. Try first, then check the answer key at the bottom.

In [ ]:
# Exercise 1: Find the 5 closest words to "teacher" and print their scores.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Which pair is MORE similar: ("rice", "bread") or ("rice", "car")?
#             Print both scores.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Use doesnt_match() to find the odd word in:
#             ["red", "blue", "green", "pencil"]
# YOUR CODE HERE

In [ ]:
# Exercise 4: Use analogy() to solve:  "walk" is to "walked" as "swim" is to ???
# YOUR CODE HERE

In [ ]:
# Exercise 5 (Challenge): Add 3 new reviews to the review list, rebuild review_vectors,
#                         and test smart_search() with a query of your own.
# YOUR CODE HERE

---
## Answer Key

In [ ]:
# Exercise 1
print("Ex 1:")
for word, score in model.most_similar("teacher", topn=5):
    print(f"  {word:<12} {score:.2f}")

# Exercise 2
print("\nEx 2:")
print("  rice & bread:", round(model.similarity("rice", "bread"), 2))
print("  rice & car:  ", round(model.similarity("rice", "car"), 2))

# Exercise 3
print("\nEx 3:", model.doesnt_match(["red", "blue", "green", "pencil"]))

# Exercise 4
print("\nEx 4: walk : walked = swim :", analogy("walk", "walked", "swim"))

# Exercise 5
reviews_plus = reviews + ["The jacket was warm and stylish.",
                          "The headphones stopped working after two days.",
                          "Customer service replied quickly and politely."]
review_vectors = np.array([sentence_vector(r) for r in reviews_plus])
reviews = reviews_plus                     # smart_search() uses the "reviews" list
print("\nEx 5:")
smart_search("fashionable coat")

---
## Great Job!

Today you:
- saw why **counting words** cannot understand meaning,
- explored real **word embeddings** with 50 numbers per word,
- drew a **map of word meanings**,
- measured **cosine similarity** and solved the odd-one-out puzzle,
- did **word math** like king − man + woman ≈ queen,
- built a **smart search tool** that searches by meaning,
- and found **bias** hidden inside the embeddings.

**Think about it:** Modern AI chatbots give the word "bank" different numbers depending on the sentence. How do you think they do it? That is our next big idea: **transformers**.